In [37]:
import pandas as pd
import numpy as np
import swifter

In [2]:
train = pd.read_csv('Data Updated/train_df.csv')
val = pd.read_csv('Data Updated/val_df.csv')
test = pd.read_csv('Data Updated/test_df.csv')

In [4]:
train.drop(columns='Unnamed: 0', inplace=True)
val.drop(columns='Unnamed: 0', inplace=True)
test.drop(columns='Unnamed: 0', inplace=True)

In [5]:
train.head()

,title,source,Date,sentiment_label,Ticker,date
0,Agilent Technologies Introduces New Version of...,Business Wire,2014-01-06,1,A,2014-01-06
1,Agilent Technologies Introduces ICP-MS and MP-...,Business Wire,2014-01-06,1,A,2014-01-06
2,Agilent Technologies Inc. Introduces New Exter...,Business Wire,2014-01-08,1,A,2014-01-08
3,AT4 Wireless Selects Agilent Technologies Test...,Business Wire,2014-01-09,1,A,2014-01-09
4,Agilent Technologies Introduces First USB 3.0 ...,Other,2014-01-09,1,A,2014-01-09


In [84]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

import string

# Download NLTK data
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')


def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
# Preprocessing function
def preprocess_text(text):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    # Tokenize text
    tokens = word_tokenize(text.lower())
    
    # Get POS tags
    pos_tags = pos_tag(tokens)

    # Remove punctuation and stopwords, and lemmatize
    tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags 
              if word not in stop_words and word not in string.punctuation]
    return ' '.join(tokens)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Bangkit\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [95]:
train['processed_title'] = train['title'].swifter.apply(preprocess_text)
val['processed_title'] = val['title'].swifter.apply(preprocess_text)
test['processed_title'] = test['title'].swifter.apply(preprocess_text)

Pandas Apply: 100%|██████████| 28387/28387 [00:38<00:00, 745.66it/s]


In [97]:

# Prepare data for Naive Bayes
X_train = train['processed_title']
y_train  = train['sentiment_label']

X_val = val['processed_title']
y_val  = val['sentiment_label']

X_test = test['processed_title']
y_test  = test['sentiment_label']

# Convert text to numerical features using CountVectorizer
vectorizer = CountVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

# Train a Naive Bayes classifier
nb = MultinomialNB()
nb.fit(X_train_vectorized, y_train)

# Make predictions
y_pred = nb.predict(X_test_vectorized)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.5035051255856554
Classification Report:
               precision    recall  f1-score   support

           0       0.52      0.47      0.50     14673
           1       0.49      0.54      0.51     13714

    accuracy                           0.50     28387
   macro avg       0.50      0.50      0.50     28387
weighted avg       0.51      0.50      0.50     28387

